# 5.0 — Data Preparation (v2)

## Changes from v1 (based on professor feedback)
- Reduced to 3 core clinical variables: UPDRS-III (OFF), MoCA, Hoehn & Yahr
- UPDRS-III filtered to OFF state only (PDSTATE == 3.0 or NaN)
- SC(Screening)↔BL(Baseline) EVENT_ID cross-mapping recovers ~119 additional patients
- Manual summation retained (NP3TOT not present in this PPMI download)
- UPDRS-IV, RBD, disease duration dropped

## 1. Imports and Configuration

In [32]:

import pandas as pd
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

In [33]:
# File paths 
LATENT_FILE    = '../../data/baseline/final_train_combined_vae_data.csv'
CLINICAL_DIR   = '../../data/raw/ppmi_clinical/'
UPDRS_III_FILE = os.path.join(CLINICAL_DIR, 'MDS-UPDRS_Part_III_06Jul2026.csv')
MOCA_FILE      = os.path.join(CLINICAL_DIR, 'Montreal_Cognitive_Assessment__MoCA__08Jul2026.csv')

TRAIN_OUT      = '../../data/processed/clinical_merged/train_v2.csv'
VAL_OUT        = '../../data/processed/clinical_merged/val_v2.csv'
SCALER_OUT     = '../../results/models/scaler_sbr_v2.pkl'
PCA_OUT        = '../../results/models/pca_sbr_v2.pkl'

os.makedirs('../../data/processed/clinical_merged', exist_ok=True)
os.makedirs('../../results/models', exist_ok=True)

# ── Parameters ─────────────────────────────────────────────────────────────────
TRAIN_RATIO    = 0.8
RANDOM_STATE   = 42
JOIN_KEY       = ['PATNO', 'EVENT_ID']
PATIENT_COL    = 'PATNO'
LABEL_COL      = 'label'
SBR_COLS = [
    'DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L',
    'DATSCAN_PUTAMEN_R', 'DATSCAN_PUTAMEN_L',
    'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT'
]
N_SBR_PCS = 3

print("Configuration loaded.")

Configuration loaded.


## 2. Load Latent Vectors

In [69]:
df_latents = pd.read_csv(LATENT_FILE)

print("Latent vectors")
print(f"Shape:           {df_latents.shape}")
print(f"Unique patients: {df_latents[PATIENT_COL].nunique()}")
print(f"Total rows:      {len(df_latents)}")
print(f"\nLabel counts:")
print(df_latents[LABEL_COL].value_counts().to_string())
print(f"\nEVENT_ID distribution (top 10):")
print(df_latents['EVENT_ID'].value_counts().head(10).to_string())

Latent vectors
Shape:           (2373, 303)
Unique patients: 1437
Total rows:      2373

Label counts:
label
PD         2030
Control     233
SWEDD       110

EVENT_ID distribution (top 10):
EVENT_ID
SC     1228
V06     392
V04     385
V10     256
U01      41
ST       32
V02      23
V05      10
U02       6


## 3. Load and Preprocess UPDRS-III
#### OFF state only

In [58]:
df_updrs3_raw = pd.read_csv(UPDRS_III_FILE)

print("UPDRS-III raw ")
print(f"Shape: {df_updrs3_raw.shape}")
print(f"\nPDSTATE distribution:")
print(df_updrs3_raw['PDSTATE'].value_counts(dropna=False).to_string())
print(f"\nEVENT_ID distribution (top 10):")
print(df_updrs3_raw['EVENT_ID'].value_counts().head(10).to_string())

# Filter to OFF state only
df_updrs3 = df_updrs3_raw[
    (df_updrs3_raw['PDSTATE'] == 'OFF') | (df_updrs3_raw['PDSTATE'].isna())
].copy()

print(f"\nAfter OFF state filter ")
print(f"Rows before: {len(df_updrs3_raw)}")
print(f"Rows after:  {len(df_updrs3)}")
print(f"  Explicit OFF (PDSTATE='OFF'): {(df_updrs3['PDSTATE']=='OFF').sum()}")
print(f"  Assumed OFF (PDSTATE=NaN):  {df_updrs3['PDSTATE'].isna().sum()}")


# Use precomputed NP3TOT
df_updrs3 = df_updrs3[JOIN_KEY + ['NP3TOT', 'NHY']].dropna(subset=['NP3TOT'])
df_updrs3 = df_updrs3.rename(columns={'NP3TOT': 'UPDRS3_TOTAL', 'NHY': 'HOEHN_YAHR'})# Rename columns for clarity

print(f"\nUPDRS-III final ")
print(f"Rows:            {len(df_updrs3)}")
print(f"Columns:            {df_updrs3.columns.tolist()}")
print(f"Unique patients: {df_updrs3['PATNO'].nunique()}")
print(f"Score range:     {df_updrs3['UPDRS3_TOTAL'].min():.0f} – {df_updrs3['UPDRS3_TOTAL'].max():.0f}")
print(f"Mean ± std:      {df_updrs3['UPDRS3_TOTAL'].mean():.1f} ± {df_updrs3['UPDRS3_TOTAL'].std():.1f}")
print(df_updrs3.head())

UPDRS-III raw 
Shape: (38626, 65)

PDSTATE distribution:
PDSTATE
NaN    21372
ON      9802
OFF     7452

EVENT_ID distribution (top 10):
EVENT_ID
BL     5382
V04    4356
V06    3771
V08    2598
V05    2239
V02    2147
V10    1832
V12    1416
SC     1171
V14    1099

After OFF state filter 
Rows before: 38626
Rows after:  28824
  Explicit OFF (PDSTATE='OFF'): 7452
  Assumed OFF (PDSTATE=NaN):  21372

UPDRS-III final 
Rows:            23405
Columns:            ['PATNO', 'EVENT_ID', 'UPDRS3_TOTAL', 'HOEHN_YAHR']
Unique patients: 5151
Score range:     0 – 100
Mean ± std:      14.2 ± 14.7
   PATNO EVENT_ID  UPDRS3_TOTAL  HOEHN_YAHR
0   3000       BL           4.0         0.0
1   3000      V04           1.0         0.0
2   3000      V06           4.0         0.0
3   3000      V08           2.0         0.0
4   3000      V10          19.0         0.0


## 4. Load MoCA

In [63]:
df_moca_raw = pd.read_csv(MOCA_FILE)

# MCATOT is the precomputed total score (0-30)
df_moca = df_moca_raw[JOIN_KEY + ['MCATOT']].dropna(subset=['MCATOT'])
df_moca = df_moca.rename(columns={'MCATOT': 'MOCA_TOTAL'})

print("MoCA")
print(f"Rows:            {len(df_moca)}")
print(f"Unique patients: {df_moca[PATIENT_COL].nunique()}")
print(f"Score range:     {df_moca['MOCA_TOTAL'].min():.0f} – {df_moca['MOCA_TOTAL'].max():.0f}")
print(f"Mean ± std:      {df_moca['MOCA_TOTAL'].mean():.1f} ± {df_moca['MOCA_TOTAL'].std():.1f}")
print(f"\nEVENT_ID distribution (top 5):")
print(df_moca_raw['EVENT_ID'].value_counts().head(5).to_string())

print(f"\nPATNO range:")
print(f"  min: {df_moca['PATNO'].min()}")
print(f"  max: {df_moca['PATNO'].max()}")
print("\nColumns: ", df_moca.columns.tolist())

MoCA
Rows:            19974
Unique patients: 5478
Score range:     0 – 30
Mean ± std:      26.8 ± 2.9

EVENT_ID distribution (top 5):
EVENT_ID
V04    3653
BL     2972
V06    2902
SC     2565
V08    1862

PATNO range:
  min: 3000
  max: 660264

Columns:  ['PATNO', 'EVENT_ID', 'MOCA_TOTAL']


In [37]:
# Moca Coverage in latent vectors
moca_max = df_moca['PATNO'].max()
missing  = df_latents[~df_latents['PATNO'].isin(df_moca['PATNO'])].drop_duplicates('PATNO')


# print(missing['PATNO'].head(5))
# print(missing['PATNO'] > moca_max)
print(f"MoCA table max PATNO: {moca_max}")
print(f"\nMissing patients total: {len(missing)}")

print(f"\n  PATNO > {moca_max} (newer cohort): {(missing['PATNO'] > moca_max).sum()}") 
print(f"  PATNO <= {moca_max} (should be covered): {(missing['PATNO'] <= moca_max).sum()}")
print(f"  Max of PATNO in latent vectors: {df_latents['PATNO'].max()}")

print(f"\nLatent vectors coverage:")
print(f"  Total latent vectors: {len(df_latents)}")
print(f"  Unique patients:      {df_latents[PATIENT_COL].nunique()}")
print(f"  Unique patients with MoCA: {df_latents[df_latents[PATIENT_COL].isin(df_moca[PATIENT_COL])][PATIENT_COL].nunique()}")  

MoCA table max PATNO: 660264

Missing patients total: 19

  PATNO > 660264 (newer cohort): 0
  PATNO <= 660264 (should be covered): 19
  Max of PATNO in latent vectors: 239077

Latent vectors coverage:
  Total latent vectors: 2373
  Unique patients:      1437
  Unique patients with MoCA: 1418


In [38]:
# Split into old and new cohort
moca_max = df_moca['PATNO'].max()  # 92,834

old_cohort = df_latents[df_latents['PATNO'] <= moca_max].drop_duplicates('PATNO')
new_cohort = df_latents[df_latents['PATNO'] >  moca_max].drop_duplicates('PATNO')

moca_patnos = set(df_moca['PATNO'].unique())

old_covered = old_cohort[old_cohort['PATNO'].isin(moca_patnos)]
old_missing = old_cohort[~old_cohort['PATNO'].isin(moca_patnos)]

print(f"Old cohort (PATNO <= {moca_max}):")
print(f"  Total patients:    {len(old_cohort)}")
print(f"  With MoCA:         {len(old_covered)}")
print(f"  Without MoCA:      {len(old_missing)}")
print(f"  Coverage:          {len(old_covered)/len(old_cohort)*100:.1f}%")

print(f"\nNew cohort (PATNO > {moca_max}):")
print(f"  Total patients:    {len(new_cohort)}")
print(f"  Coverage:          0% (not in MoCA download)")

Old cohort (PATNO <= 660264):
  Total patients:    1437
  With MoCA:         1418
  Without MoCA:      19
  Coverage:          98.7%

New cohort (PATNO > 660264):
  Total patients:    0
  Coverage:          0% (not in MoCA download)


## 5. Merge Clinical Data with Latent Vectors
**SC(Screening)↔BL(Baseline) cross-mapping fix:**  
At enrollment, DaTSCAN is often acquired at EVENT_ID = SC while UPDRS/MoCA  
are measured at EVENT_ID = BL (or vice versa). A two-step merge recovers  
these patients: first exact match, then SC↔BL remapped match for unmatched rows.


### Fix Event ID Matching

In [78]:
# [MERGE-WITH-SCBL-FIX]

def merge_with_scbl_fix(df_base, df_clinical, clinical_cols, label):
    """
    Two-step merge:
    1. Exact match on PATNO + EVENT_ID
    2. SC<->BL remap for unmatched rows
    """
    # Reset index to avoid multi-index issues
    df_base = df_base.reset_index(drop=True)
    df_clinical = df_clinical.reset_index(drop=True)

    # Step 1 — exact match
    merged = df_base.merge(
        df_clinical[JOIN_KEY + clinical_cols],
        on=JOIN_KEY,
        how='left'
    )

    # Step 2 — SC-BL remap for rows still missing clinical data
    missing_mask = merged[clinical_cols[0]].isna()
    n_missing_before = missing_mask.sum()

    if missing_mask.sum() > 0:
        sc_bl_map = {'SC': 'BL', 'BL': 'SC'}
        df_clinical_remapped = df_clinical.copy()
        df_clinical_remapped['EVENT_ID'] = df_clinical_remapped['EVENT_ID'].replace(sc_bl_map)

        # Merge only the missing rows with remapped clinical table
        missing_rows = df_base[missing_mask].copy()
        fallback = missing_rows.merge(
            df_clinical_remapped[JOIN_KEY + clinical_cols],
            on=JOIN_KEY,
            how='left'
        )

        # Fill missing values from fallback
        for col in clinical_cols:
            merged.loc[missing_mask, col] = fallback[col].values

    n_missing_after = merged[clinical_cols[0]].isna().sum()
    n_recovered = n_missing_before - n_missing_after

    print(f"  {label}:")
    print(f"    Matched (exact):    {(~merged[clinical_cols[0]].isna()).sum()- n_recovered}")
    print(f"    Recovered (SC-BL): {n_recovered}")
    print(f"    Total matched:      {(~merged[clinical_cols[0]].isna()).sum()} / {len(df_base)} rows ({(~merged[clinical_cols[0]].isna()).sum()/len(df_base)*100:.1f}%)")
    print()
    # print(merged.head())
    return merged

print("=== Merging clinical tables ===\n")

df = df_latents.copy().reset_index(drop=True)
print(f"Initial shape: {df.shape}")

# Merge UPDRS-III and Hoehn & Yahr
df = merge_with_scbl_fix(df, df_updrs3, ['UPDRS3_TOTAL', 'HOEHN_YAHR'], 'UPDRS-III + H&Y')

# Merge MoCA
df = merge_with_scbl_fix(df, df_moca, ['MOCA_TOTAL'], 'MoCA')

print(f"Final shape: {df.shape}")

=== Merging clinical tables ===

Initial shape: (2373, 303)
  UPDRS-III + H&Y:
    Matched (exact):    1393
    Recovered (SC-BL): 567
    Total matched:      1960 / 2373 rows (82.6%)

  MoCA:
    Matched (exact):    2257
    Recovered (SC-BL): 5
    Total matched:      2262 / 2373 rows (95.3%)

Final shape: (2373, 306)
